In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from time import sleep

In [5]:
df = pd.read_csv("SNAP Member Info - Sheet1.csv")
locations = df[["City", "State"]].dropna().drop_duplicates()

# Split rows where City and State contain "/" into separate rows
split_rows = []
keep_mask = []
for idx, row in locations.iterrows():
    if "/" in str(row["City"]):
        cities = [c.strip() for c in row["City"].split("/")]
        states = [s.strip() for s in row["State"].split("/")]
        for city, state in zip(cities, states):
            split_rows.append({"City": city, "State": state})
        keep_mask.append(False)
    else:
        keep_mask.append(True)

locations = pd.concat(
    [locations[keep_mask], pd.DataFrame(split_rows)],
    ignore_index=True
).drop_duplicates()
locations

,City,State
0,Arlington,VA
1,Iowa City,IA
2,Birmingham,Al
3,Chapel Hill,NC
4,Baltimore,MD
5,Los Angeles,CA
6,Athens,GA
7,Cambridge,MA
8,Ithaca,NY
9,New Haven,CT


In [7]:
geolocator = Nominatim(user_agent="snap_member_map")

def get_coords(row):
    query = f"{row['City']}, {row['State']}, USA"
    location = geolocator.geocode(query)
    sleep(1)  # respect Nominatim rate limit
    if location:
        return pd.Series({"lat": location.latitude, "lon": location.longitude})
    return pd.Series({"lat": None, "lon": None})

locations[["lat", "lon"]] = locations.apply(get_coords, axis=1)
locations

,City,State,lat,lon
0,Arlington,VA,38.876933,-77.089309
1,Iowa City,IA,41.661256,-91.529911
2,Birmingham,Al,33.520682,-86.802433
3,Chapel Hill,NC,35.913154,-79.055780
4,Baltimore,MD,39.290882,-76.610759
5,Los Angeles,CA,34.053691,-118.242766
6,Athens,GA,33.959768,-83.376398
7,Cambridge,MA,42.365635,-71.104002
8,Ithaca,NY,42.437418,-76.548372
9,New Haven,CT,41.308214,-72.925052


In [9]:
for _, row in locations.iterrows():
    print(f"[[{row['lat']:.6f}, {row['lon']:.6f}], '{row['City']}, {row['State']}'],")

[[38.876933, -77.089309], 'Arlington, VA'],
[[41.661256, -91.529911], 'Iowa City, IA'],
[[33.520682, -86.802433], 'Birmingham, Al'],
[[35.913154, -79.055780], 'Chapel Hill, NC'],
[[39.290882, -76.610759], 'Baltimore, MD'],
[[34.053691, -118.242766], 'Los Angeles, CA'],
[[33.959768, -83.376398], 'Athens, GA'],
[[42.365635, -71.104002], 'Cambridge, MA'],
[[42.437418, -76.548372], 'Ithaca, NY'],
[[41.308214, -72.925052], 'New Haven, CT'],
[[37.870839, -122.272863], 'Berkeley, CA'],
[[39.952724, -75.163526], 'Philadelphia, PA'],
[[29.651968, -82.324985], 'Gainesville, FL'],
[[39.739236, -104.984862], 'Denver, CO'],
[[47.603832, -122.330062], 'Seattle, WA'],
[[43.074690, -89.384166], 'Madison, WI'],
[[40.014986, -105.270545], 'Boulder, CO'],
[[37.787936, -122.407520], 'San Francisco, CA'],
[[38.625406, -90.190009], 'St. Louis, MO'],
[[38.980666, -76.936919], 'College Park, MD'],
[[25.774157, -80.193597], 'Miami, FL'],
[[42.331551, -83.046640], 'Detroit, MI'],
[[37.444329, -122.159847], 'Pal